In [1]:
# Importa biblioteca 

import geopandas as gpd 
import folium 
import pandas as pd 
from shapely.geometry import box 
import pyproj


In [2]:
# Carregar dados  

gdf_uc = gpd.read_file(r"C:\cnuc_2026_03_atualizado.shp", encoding='cp1252')
print (gdf_uc)
print (gdf_uc.crs)

c:\anaconda\envs\pygis\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: One or several characters couldn't be converted correctly from cp1252 to UTF-8. This warning will not be emitted anymore
  return ogr_read(


     gml_id docleg_id  uc_id       cd_cnuc   wdpa_pid  \
0      None        16     11  0000.00.0011  555600306   
1      None       403    275  0000.00.0275     351823   
2      None       925    981  0000.13.0981     352134   
3      None       928    984  0000.13.0984     352142   
4      None       929    985  0000.13.0985     352139   
...     ...       ...    ...           ...        ...   
3242   None     67859  25337  0000.00.0150     351828   
3243   None     67872  25340  0000.31.5329        NaN   
3244   None     67952  25367  0000.00.5332        NaN   
3245   None     67992  25375  0000.00.0175       2581   
3246   None     68017  25379  0000.00.0062       2220   

                                                nome_uc    cria_ano  \
0     ÃREA DE PROTEÃ‡ÃƒO AMBIENTAL SERRA DA MANTIQUEIRA  06-06-1985   
1                        RESERVA BIOLÃ“GICA DAS PEROBAS  21-03-2006   
2         RESERVA DE DESENVOLVIMENTO SUSTENTÃVEL AMANÃƒ  06-08-1998   
3        RESERVA DE DESENVOLVIM

In [ ]:
# Criando a Bounding Box - definir os limites da caixa (exemplo: uma área específica) caso você já tenha (Caso não, pule a celula)

# minx, miny = -47.5, -22.5
# maxx, maxy = -46.5, -21.5

# bbox_geom = box(minx, miny, maxx, maxy)

# bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs="EPSG:4326")
# bbox_gdf = bbox_gdf.to_crs(gdf_uc.crs)


Foram criados dois Html para essa análise, o primeiro você ira delimitar um poligono de seu interesse e exporta para usar no segundo, ou , você pode colocar uma box ja definida no quadrante acima, com o geojson da área você consegue chegar no objetivo.

Objetivo é responder perguntas como:
- Quanto da minha área sobrepõe UCs?
- Existe essa sobreposição? quantos hectares?
- Quantos hectares da minha área estão dentro de UCs?
- Qual porcentagem da minha área está em UCs?

In [ ]:
# Criando um Folium para desenhar o poligono que vai ser usado no outro folium visualização 

from folium.plugins import Draw, Geocoder


# GARANTIR CRS CORRETO

if gdf_uc.crs is None:
    gdf_uc = gdf_uc.set_crs("EPSG:4326")

# reprojetar para cálculo 
gdf_uc = gdf_uc.to_crs("EPSG:5880")


# VOLTAR PRA VISUALIZAÇÃO

gdf_uc_web = gdf_uc.to_crs("EPSG:4326")

# corrigir geometria (Não precisou, porém por segurança deixei)
gdf_uc_web["geometry"] = gdf_uc_web.buffer(0)

# (opcional, melhora performance)
# gdf_uc_web["geometry"] = gdf_uc_web.simplify(0.01)


#  MAPA

m = folium.Map(
    location=[-15, -55],
    zoom_start=4,
    tiles="Esri.WorldImagery"
)

# barra de busca
Geocoder().add_to(m)

# UCs
folium.GeoJson(
    gdf_uc_web,
    name="UCs"
).add_to(m)

# ferramenta de desenho
draw = Draw(
    export=True,
    draw_options={
        "rectangle": True,
        "polygon": True,
        "circle": False,
        "marker": False,
        "polyline": False
    },
    edit_options={"edit": True}
)

draw.add_to(m)

# salvar
m.save("mapa_desenho.html")




In [ ]:



# CARREGAR DADOS você pode colocar um shp de um CAR 

gdf_draw = gpd.read_file(r"C:data.geojson")

# GARANTIR CRS

print("CRS UCs:", gdf_uc.crs)
print("CRS desenho:", gdf_draw.crs)

if gdf_draw.crs is None:
    gdf_draw = gdf_draw.set_crs("EPSG:4326")  # EPSG para visualização  lat/long


#  CORRIGIR GEOMETRIA

gdf_uc["geometry"] = gdf_uc.buffer(0)
gdf_draw["geometry"] = gdf_draw.buffer(0)


# REPROJETAR PARA CÁLCULO

gdf_uc_proj = gdf_uc.to_crs("EPSG:5880")
gdf_draw_proj = gdf_draw.to_crs("EPSG:5880")  # EPSG para cálculo m


# INTERSEÇÃO

intersection = gpd.overlay(gdf_uc_proj, gdf_draw_proj, how="intersection")

print("Interseções encontradas:", len(intersection))


# CÁLCULO DE ÁREA

if not intersection.empty:
    intersection["area_ha"] = intersection.area / 10000
    draw_area = gdf_draw_proj.area.sum() / 10000
    intersection["percent"] = (intersection["area_ha"] / draw_area) * 100


#  VOLTAR PRA VISUALIZAÇÃO

gdf_uc_web = gdf_uc.to_crs("EPSG:4326")
gdf_draw_web = gdf_draw.to_crs("EPSG:4326")

# simplificação leve (opcional)
gdf_uc_web["geometry"] = gdf_uc_web.simplify(0.001)

if not intersection.empty:
    intersection_web = intersection.to_crs("EPSG:4326")


# 8. CENTRALIZAR 

centro = gdf_draw_proj.geometry.centroid.iloc[0]

# converter centro pra lat/long
centro_latlon = gpd.GeoSeries([centro], crs="EPSG:5880").to_crs("EPSG:4326").iloc[0]

m = folium.Map(
    location=[centro_latlon.y, centro_latlon.x],
    zoom_start=8,
    tiles="Esri.WorldImagery"
)


# CAMADAS - uma para as UCs, outraa foca no poligono e por ultimo a interseção entre eles


# UCs
folium.GeoJson(
    gdf_uc_web,
    name="Unidades de Conservação",
    style_function=lambda x: {
        "color": "blue",
        "weight": 1,
        "fillOpacity": 0.1
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["nome_uc"],
        aliases=["UC:"]
    )
).add_to(m)

# desenho
folium.GeoJson(
    gdf_draw_web,
    name="Área desenhada",
    style_function=lambda x: {
        "color": "red",
        "weight": 2
    }
).add_to(m)

# interseção
if not intersection.empty:
    folium.GeoJson(
        intersection_web,
        name="Sobreposição",
        style_function=lambda x: {
            "color": "green",
            "weight": 2
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["nome_uc", "area_ha", "percent"],
            aliases=["UC:", "Área (ha):", "%:"],
            localize=True
        )
    ).add_to(m)

folium.LayerControl().add_to(m)


#  SALVAR

m.save("index.html")





CRS UCs: EPSG:5880
CRS desenho: EPSG:4326
Interseções encontradas: 8
